
# Sports Betting Fraud Detection

**Group Members:** Aryaman Arora (aa833), Hyunjin Lee (hl494), Suhaib Mansour (sm1097), Emily Zhao (egz5)



In [1]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import difflib
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


## Load and Clean Data

In [2]:

def standardize_team_names(df, col, canonical_names, cutoff=0.7):
    def match_name(name):
        matches = difflib.get_close_matches(str(name), canonical_names, n=1, cutoff=cutoff)
        return matches[0] if matches else name
    df[col] = df[col].apply(match_name)
    return df

def clean_betting_data(match_df, odds_df):
    match_df['Date'] = pd.to_datetime(match_df['Date'], errors='coerce')
    odds_df['Date'] = pd.to_datetime(odds_df['Date'], errors='coerce')
    match_df = match_df.dropna(subset=['HomeTeam', 'AwayTeam'])
    match_df = match_df.drop_duplicates(subset=['HomeTeam', 'AwayTeam', 'Date'])
    merged = pd.merge(match_df, odds_df, on=['Date', 'HomeTeam', 'AwayTeam'], how='inner')
    merged['delta_odds'] = merged['ClosingOdds'] - merged['OpeningOdds']
    merged['odds_volatility'] = merged[['OpeningOdds', 'ClosingOdds']].std(axis=1)
    merged['odds_pct_change'] = (merged['delta_odds'] / merged['OpeningOdds']).fillna(0)
    numeric_cols = ['delta_odds', 'odds_volatility', 'odds_pct_change']
    merged[numeric_cols] = merged[numeric_cols].fillna(merged[numeric_cols].median())
    return merged


## Baseline Labels

In [3]:

def add_features_and_labels(df):
    scaler = MinMaxScaler()
    features = ['delta_odds', 'odds_volatility', 'odds_pct_change']
    df[[f + '_scaled' for f in features]] = scaler.fit_transform(df[features])
    threshold = df['delta_odds'].abs().quantile(0.95)
    df['suspicious'] = (df['delta_odds'].abs() >= threshold).astype(int)
    return df


##  graphs/distributions

In [4]:

def plot_distributions(df):
    plt.figure(figsize=(8, 5))
    sns.histplot(df['delta_odds'], bins=30, kde=True)
    plt.title("Distribution of Odds Change (ΔOdds)")
    plt.xlabel("ΔOdds")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.boxplot(x='suspicious', y='odds_volatility', data=df)
    plt.title("Volatility by Suspicious Label")
    plt.show()


## the main ML model

In [5]:

def train_baseline_model(df):
    X = df[['delta_odds', 'odds_volatility', 'odds_pct_change']]
    y = df['suspicious']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("=== Random Forest Baseline Model ===")
    print(classification_report(y_test, y_pred))
    feature_importances = pd.Series(model.feature_importances_, index=X.columns)
    feature_importances.plot(kind='bar', title='Feature Importances')
    plt.show()
    return model


## execution

In [6]:

import kagglehub
import os


path = kagglehub.dataset_download("eladsil/football-games-odds")
print("files at", path)


files = [f for f in os.listdir(path) if f.endswith(".csv")]
print("Available CSV files:", files)


file_path = os.path.join(path, files[0])  
print("Using file:", file_path)

# Read dataset
df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
df.head()


rename_map = {
    'HomeTeam': 'HomeTeam',
    'AwayTeam': 'AwayTeam',
    'Date': 'Date',
    'B365H': 'OpeningOdds',  
    'B365A': 'ClosingOdds'  
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})


df = df.dropna(subset=['HomeTeam', 'AwayTeam', 'OpeningOdds', 'ClosingOdds'])


match_df = df[['Date', 'HomeTeam', 'AwayTeam']].copy()
odds_df = df[['Date', 'HomeTeam', 'AwayTeam', 'OpeningOdds', 'ClosingOdds']].copy()


merged = clean_betting_data(match_df, odds_df)
merged = add_features_and_labels(merged)


plot_distributions(merged)
train_baseline_model(merged)


merged.to_csv("Cleaned_Football_Odds.csv", index=False)
print("Cleaned dataset saved as Cleaned_Football_Odds.csv")


100%|██████████| 5.06M/5.06M [00:00<00:00, 20.9MB/s]

Extracting files...


files at /Users/aryamanarora/.cache/kagglehub/datasets/eladsil/football-games-odds/versions/2
Available CSV files: ['Matches_Odds.csv', 'Matches_Results.csv']
Using file: /Users/aryamanarora/.cache/kagglehub/datasets/eladsil/football-games-odds/versions/2/Matches_Odds.csv
Dataset shape: (461144, 9)


KeyError: ['HomeTeam', 'AwayTeam', 'OpeningOdds', 'ClosingOdds']